In [1]:
import pandas as pd
import os
from datetime import datetime, timedelta


directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2022/'

# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

# monthly data
monthly_data = []

# year (2022)
year = 2022

 # Loop monthly
for month in range(1, 13): 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:  # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
            # Gå i stå, slet tomme filer
            if os.path.getsize(message_file) == 0 or os.path.getsize(orderbook_file) == 0:
                print(f"Skipping empty file: {message_file} or {orderbook_file}")
                continue  

            try:
        
                message_df = pd.read_csv(message_file, encoding='utf-8', low_memory=False)
                orderbook_df = pd.read_csv(orderbook_file, encoding='utf-8', low_memory=False)
            except Exception as e:
                print(f"Error loading file {filename}: {e}")
                continue  # Skip this file if there's an error

            # Dropper col 7
            message_df = message_df.iloc[:, :-1]  # Drop last column
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            # Start dag SPY_2022-01-03
            base_date = filename.split('_')[1]
            
            
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merge message and orderbook 
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Slet NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' as index for 1 sec
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            
            monthly_files.append(resampled_df)
    
    # Concatenate all daily files for the month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)


final_df = pd.concat(monthly_data)


final_df.to_csv(f'combined_SPY_{year}_cleaned.csv', index=False)

print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2022-01-03 09:30:00    4763400.0       710.0    4763000.0      1000.0   
2022-01-03 09:30:01    4763500.0       290.0    4763200.0       500.0   
2022-01-03 09:30:02    4763000.0       109.0    4762700.0      2000.0   
2022-01-03 09:30:03    4763300.0       302.0    4763000.0       660.0   
2022-01-03 09:30:04    4763300.0         2.0    4763200.0        17.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2022-01-03 09:30:00    4763500.0       500.0    4762800.0       500.0   
2022-01-03 09:30:01    4763600.0       502.0    4763100.0       774.0   
2022-01-03 09:30:02    4763100.0       108.0    4762600.0      1100.0   
2022-01-03 09:30:03    4763400.0       600.0    4762900.0       100.0   
2022-01-03 09:30:04    4763400.0      1646.0    47

In [2]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2022-12-30 15:59:55    3824600.0        40.0    3824500.0      1000.0   
2022-12-30 15:59:56    3824800.0       300.0    3824600.0      1300.0   
2022-12-30 15:59:57    3825000.0       200.0    3824800.0      1000.0   
2022-12-30 15:59:58    3824400.0       100.0    3824200.0      1100.0   
2022-12-30 15:59:59    3824700.0       178.0    3824600.0       500.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2022-12-30 15:59:55    3824700.0       400.0    3824400.0       600.0   
2022-12-30 15:59:56    3824900.0       200.0    3824500.0      1000.0   
2022-12-30 15:59:57    3825100.0       700.0    3824700.0      3800.0   
2022-12-30 15:59:58    3824500.0       200.0    3824100.0      2700.0   
2022-12-30 15:59:59    3824800.0       100.0    38

In [3]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5499911


In [4]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2022-01-03 09:30:00,4763400.0,710.0,4763000.0,1000.0,4763500.0,500.0,4762800.0,500.0,4.0,34391160.0,500.0,4763000.0,1.0
2022-01-03 09:30:01,4763500.0,290.0,4763200.0,500.0,4763600.0,502.0,4763100.0,774.0,4.0,34738000.0,2.0,4763500.0,-1.0
2022-01-03 09:30:02,4763000.0,109.0,4762700.0,2000.0,4763100.0,108.0,4762600.0,1100.0,1.0,35196916.0,1000.0,4762700.0,1.0
2022-01-03 09:30:03,4763300.0,302.0,4763000.0,660.0,4763400.0,600.0,4762900.0,100.0,3.0,35411196.0,100.0,4762900.0,1.0
2022-01-03 09:30:04,4763300.0,2.0,4763200.0,17.0,4763400.0,1646.0,4763100.0,2.0,1.0,35582952.0,1000.0,4763400.0,-1.0


In [5]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2022-12-30 15:59:55,3824600.0,40.0,3824500.0,1000.0,3824700.0,400.0,3824400.0,600.0,3.0,814459652.0,100.0,3824700.0,-1.0
2022-12-30 15:59:56,3824800.0,300.0,3824600.0,1300.0,3824900.0,200.0,3824500.0,1000.0,1.0,814628504.0,100.0,3824800.0,-1.0
2022-12-30 15:59:57,3825000.0,200.0,3824800.0,1000.0,3825100.0,700.0,3824700.0,3800.0,1.0,814737596.0,100.0,3824800.0,1.0
2022-12-30 15:59:58,3824400.0,100.0,3824200.0,1100.0,3824500.0,200.0,3824100.0,2700.0,3.0,814887004.0,300.0,3824400.0,-1.0
2022-12-30 15:59:59,3824700.0,178.0,3824600.0,500.0,3824800.0,100.0,3824500.0,1100.0,3.0,815030160.0,1000.0,3824800.0,-1.0


In [6]:
# Save 
final_df.to_csv('final_combined_2022.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2022_with_time.csv', index=True)

In [7]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 247


In [8]:
# Load the data from 'final_combined_2022_with_time.csv'
final_combined_2022_with_time = pd.read_csv('final_combined_2022_with_time.csv')


final_combined_2022_with_time['Time (sec)'] = pd.to_datetime(final_combined_2022_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2022_with_time.groupby(final_combined_2022_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate through each group each day
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [9]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2022-01-03 09:30:00          4763400.0        4771700.0        4763000.0   
1 2022-01-03 09:35:00          4770800.0        4777200.0        4769200.0   
2 2022-01-03 09:40:00          4773000.0        4774500.0        4770300.0   
3 2022-01-03 09:45:00          4771800.0        4772300.0        4745800.0   
4 2022-01-03 09:50:00          4749100.0        4754800.0        4738700.0   
5 2022-01-03 09:55:00          4750400.0        4755100.0        4745600.0   
6 2022-01-03 10:00:00          4749400.0        4757400.0        4747900.0   
7 2022-01-03 10:05:00          4752800.0        4754900.0        4749200.0   
8 2022-01-03 10:10:00          4752200.0        4761800.0        4752200.0   
9 2022-01-03 10:15:00          4760800.0        4764300.0        4758200.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         4771200.0          4763000.0        4771600.0        47

In [10]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2022_corrected.csv', index=False)

In [11]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()


print(observations_per_day)

Date
2022-01-03    78
2022-01-04    78
2022-01-05    78
2022-01-06    78
2022-01-07    78
              ..
2022-12-23    78
2022-12-27    78
2022-12-28    78
2022-12-29    78
2022-12-30    78
Length: 247, dtype: int64


In [12]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2022_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2022-01-03 09:30:00          4763400.0        4771700.0        4763000.0   
1 2022-01-03 09:35:00          4770800.0        4777200.0        4769200.0   
2 2022-01-03 09:40:00          4773000.0        4774500.0        4770300.0   
3 2022-01-03 09:45:00          4771800.0        4772300.0        4745800.0   
4 2022-01-03 09:50:00          4749100.0        4754800.0        4738700.0   
5 2022-01-03 09:55:00          4750400.0        4755100.0        4745600.0   
6 2022-01-03 10:00:00          4749400.0        4757400.0        4747900.0   
7 2022-01-03 10:05:00          4752800.0        4754900.0        4749200.0   
8 2022-01-03 10:10:00          4752200.0        4761800.0        4752200.0   
9 2022-01-03 10:15:00          4760800.0        4764300.0        4758200.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         4771200.0          4763000.0        4771600.0        47

In [13]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2022_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2022-01-03 09:30:00,4763400.0,4771700.0,4763000.0,4771200.0,4763000.0,4771600.0,4762700.0,4771000.0,4763500.0,4771800.0,4763100.0,4771300.0,4762800.0,4771500.0,4762600.0,4770900.0,106100.0,353.666667,91203.0,304.010000,128187.0,427.290000,136933.0,456.443333,4763000.0,4771800.0,4762700.0,4771200.0,0.053333,2022-01-03
2022-01-03 09:35:00,4770800.0,4777200.0,4769200.0,4772200.0,4770700.0,4777000.0,4769000.0,4772100.0,4770900.0,4777300.0,4769300.0,4772300.0,4770600.0,4776900.0,4768900.0,4772000.0,150174.0,500.580000,108430.0,361.433333,161841.0,539.470000,158896.0,529.653333,4770700.0,4777100.0,4769000.0,4772200.0,-0.053333,2022-01-03
2022-01-03 09:40:00,4773000.0,4774500.0,4770300.0,4770800.0,4772700.0,4774300.0,4770200.0,4770600.0,4773100.0,4774600.0,4770400.0,4770900.0,4772600.0,4774200.0,4770100.0,4770500.0,104502.0,348.340000,100968.0,336.560000,142096.0,473.653333,157497.0,524.990000,4772800.0,4774300.0,4770300.0,4770800.0,0.053333,2022-01-03
2022-01-03 09:45:00,4771800.0,4772300.0,4745800.0,4748200.0,4771600.0,4772100.0,4745600.0,4747900.0,4771900.0,4772400.0,4745900.0,4748300.0,4771500.0,4772000.0,4745500.0,4747800.0,79473.0,264.910000,106053.0,353.510000,133424.0,444.746667,154982.0,516.606667,4771900.0,4772100.0,4745600.0,4748200.0,0.126667,2022-01-03
2022-01-03 09:50:00,4749100.0,4754800.0,4738700.0,4750300.0,4749000.0,4754500.0,4738600.0,4750100.0,4749200.0,4754900.0,4738800.0,4750400.0,4748900.0,4754400.0,4738500.0,4750000.0,86557.0,288.523333,86587.0,288.623333,134031.0,446.770000,127468.0,424.893333,4749000.0,4754800.0,4738600.0,4750100.0,-0.120000,2022-01-03
2022-01-03 09:55:00,4750400.0,4755100.0,4745600.0,4749400.0,4750200.0,4754900.0,4745400.0,4749300.0,4750500.0,4755200.0,4745700.0,4749500.0,4750100.0,4754800.0,4745300.0,4749200.0,77149.0,257.163333,112624.0,375.413333,128008.0,426.693333,140869.0,469.563333,4750400.0,4755100.0,4745600.0,4749400.0,0.013333,2022-01-03
2022-01-03 10:00:00,4749400.0,4757400.0,4747900.0,4752300.0,4749200.0,4757100.0,4747700.0,4752100.0,4749500.0,4757500.0,4748000.0,4752400.0,4749100.0,4757000.0,4747600.0,4752000.0,105641.0,352.136667,81808.0,272.693333,143164.0,477.213333,128633.0,428.776667,4749400.0,4757400.0,4747900.0,4752100.0,0.013333,2022-01-03
2022-01-03 10:05:00,4752800.0,4754900.0,4749200.0,4752200.0,4752600.0,4754800.0,4749100.0,4752000.0,4752900.0,4755000.0,4749300.0,4752300.0,4752500.0,4754700.0,4749000.0,4751900.0,77637.0,258.790000,86722.0,289.073333,145468.0,484.893333,142919.0,476.396667,4752800.0,4754800.0,4749100.0,4752000.0,0.080000,2022-01-03
2022-01-03 10:10:00,4752200.0,4761800.0,4752200.0,4760700.0,4752000.0,4761700.0,4752000.0,4760500.0,4752300.0,4761900.0,4752300.0,4760800.0,4751900.0,4761600.0,4751900.0,4760400.0,130912.0,436.373333,83694.0,278.980000,167646.0,558.820000,128122.0,427.073333,4752200.0,4761800.0,4752200.0,4760700.0,0.006667,2022-01-03
2022-01-03 10:15:00,4760800.0,4764300.0,4758200.0,4761900.0,4760600.0,4764100.0,4758000.0,4761800.0,4760900.0,4764400.0,4758300.0,4762000.0,4760500.0,4764000.0,4757900.0,4761700.0,118899.0,396.330000,83705.0,279.016667,175515.0,585.050000,151976.0,506.586667,4760800.0,4764200.0,4758200.0,4761800.0,0.026667,2022-01-03


In [14]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.tail(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2022_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2022-12-30 15:10:00,3798800.0,3806600.0,3798100.0,3804800.0,3798600.0,3806500.0,3797900.0,3804700.0,3799000.0,3806700.0,3798200.0,3804900.0,3798500.0,3806400.0,3797800.0,3804600.0,114523.0,381.743333,164906.0,549.686667,205232.0,684.106667,269515.0,898.383333,3798800.0,3806600.0,3798000.0,3804700.0,0.040000,2022-12-30
2022-12-30 15:15:00,3804700.0,3810200.0,3804700.0,3805500.0,3804600.0,3810000.0,3804600.0,3805300.0,3804800.0,3810300.0,3804800.0,3805600.0,3804500.0,3809900.0,3804500.0,3805200.0,132344.0,441.146667,188239.0,627.463333,211780.0,705.933333,295437.0,984.790000,3804650.0,3810200.0,3804650.0,3805300.0,0.080000,2022-12-30
2022-12-30 15:20:00,3805500.0,3808300.0,3804200.0,3805200.0,3805300.0,3808000.0,3804100.0,3805000.0,3805600.0,3808400.0,3804300.0,3805300.0,3805200.0,3807900.0,3804000.0,3804900.0,118676.0,395.586667,184601.0,615.336667,167985.0,559.950000,334420.0,1114.733333,3805500.0,3808300.0,3804000.0,3805000.0,0.040000,2022-12-30
2022-12-30 15:25:00,3805200.0,3805600.0,3802300.0,3803100.0,3805000.0,3805400.0,3802100.0,3803000.0,3805300.0,3805700.0,3802400.0,3803200.0,3804900.0,3805300.0,3802000.0,3802900.0,119548.0,399.826087,153444.0,513.190635,202630.0,677.692308,286856.0,959.384615,3804900.0,3805600.0,3802200.0,3803000.0,0.096990,2022-12-30
2022-12-30 15:30:00,3803100.0,3810700.0,3802000.0,3809900.0,3803000.0,3810500.0,3801900.0,3809800.0,3803200.0,3810800.0,3802100.0,3810100.0,3802900.0,3810400.0,3801800.0,3809700.0,142193.0,473.976667,145911.0,486.370000,242348.0,807.826667,279196.0,930.653333,3803100.0,3810500.0,3802000.0,3809700.0,0.060000,2022-12-30
2022-12-30 15:35:00,3810200.0,3811200.0,3807900.0,3810500.0,3809900.0,3811000.0,3807800.0,3810400.0,3810300.0,3811300.0,3808000.0,3810600.0,3809800.0,3810900.0,3807600.0,3810300.0,116928.0,389.760000,147147.0,490.490000,226052.0,753.506667,239160.0,797.200000,3809900.0,3811200.0,3807900.0,3810500.0,0.080000,2022-12-30
2022-12-30 15:40:00,3810500.0,3813000.0,3809100.0,3811300.0,3810400.0,3812800.0,3809000.0,3811100.0,3810600.0,3813100.0,3809200.0,3811400.0,3810300.0,3812700.0,3808900.0,3811000.0,162301.0,541.003333,136476.0,454.920000,307813.0,1026.043333,211198.0,703.993333,3810400.0,3812900.0,3808900.0,3811000.0,0.060000,2022-12-30
2022-12-30 15:45:00,3811300.0,3819700.0,3810700.0,3819700.0,3811100.0,3819400.0,3810500.0,3819400.0,3811400.0,3819800.0,3810800.0,3819800.0,3811000.0,3819300.0,3810400.0,3819300.0,131018.0,436.726667,93603.0,312.010000,258661.0,862.203333,150507.0,501.690000,3811100.0,3819600.0,3810700.0,3819500.0,0.086667,2022-12-30
2022-12-30 15:50:00,3819400.0,3823900.0,3819400.0,3823300.0,3818900.0,3823800.0,3818900.0,3823100.0,3819500.0,3824000.0,3819500.0,3823400.0,3818800.0,3823700.0,3818800.0,3823000.0,129374.0,431.246667,82876.0,276.253333,210024.0,700.080000,147487.0,491.623333,3818900.0,3823900.0,3818900.0,3823100.0,0.286667,2022-12-30
2022-12-30 15:55:00,3823100.0,3825000.0,3820500.0,3824700.0,3823000.0,3824800.0,3820300.0,3824600.0,3823200.0,3825100.0,3820600.0,3824800.0,3822900.0,3824700.0,3820200.0,3824500.0,145111.0,483.703333,139344.0,464.480000,321666.0,1072.220000,328199.0,1093.996667,3823000.0,3825000.0,3820500.0,3824800.0,0.006667,2022-12-30
